# MossFormer

In [ ]:
import warnings
from pathlib import Path
from tqdm import tqdm
import numpy as np
import soundfile as sf
from scipy import signal
import torch
from utils.clear_memory import clear_memory
from utils.batch import batch_denoise

# Suppress warnings
warnings.filterwarnings('ignore')

In [ ]:
from utils.config import get_audio_files, get_output_dir

files_Pitt = get_audio_files('Pitt-origin')
out_Pitt = get_output_dir('Pitt-origin', 'MossFormer')

files_Lu = get_audio_files('Lu')
out_Lu = get_output_dir('Lu', 'MossFormer')

## Load MossFormer Model

In [ ]:
from clearvoice import ClearVoice

# Use MossFormer model
model_name = 'MossFormerGAN_SE_16K'
target_sr = 16000  # Target sample rate

myClearVoice = ClearVoice(
    task='speech_enhancement',
    model_names=[model_name]
)

## Denoise Function

In [ ]:
def denoise_audio(audio_path, model, target_sr=16000):
    """
    Apply MossFormer for speech denoising and enhancement

    Args:
        audio_path: Input audio file path
        model: ClearVoice model instance
        target_sr: Target sample rate (16000)

    Returns:
        denoised_audio: Denoised audio numpy array
        sr: Sample rate
    """
    # Load audio
    audio, sr = sf.read(str(audio_path))

    # If multi-channel, convert to mono first (before resampling)
    if len(audio.shape) == 2:
        audio = np.mean(audio, axis=1)

    # Resample to target sample rate (using scipy, more stable)
    if sr != target_sr:
        num_samples = int(len(audio) * target_sr / sr)
        audio = signal.resample(audio, num_samples)

    # Ensure float32 type
    audio = audio.astype(np.float32)

    # Convert to [batch, length] format
    audio = np.reshape(audio, [1, audio.shape[0]])

    # Apply MossFormer denoising
    with torch.no_grad():
        output_wav = model(audio, online_write=False)

    # output_wav shape: [batch, length]
    return output_wav[0, :], target_sr

## Pitt Denoise

In [ ]:
denoise_fn = lambda p: denoise_audio(p, myClearVoice, target_sr)

clear_memory()

batch_denoise(
    files_Pitt['Dementia'],
    out_Pitt / 'Dementia',
    denoise_fn,
    'Dementia',
)

clear_memory()

batch_denoise(
    files_Pitt['Control'],
    out_Pitt / 'Control',
    denoise_fn,
    'Control',
)

## Lu Denoise

In [ ]:
# denoise_fn = lambda p: denoise_audio(p, myClearVoice, target_sr)

# clear_memory()

# batch_denoise(
#     files_Lu['Dementia'],
#     out_Lu / 'Dementia',
#     denoise_fn,
#     'Dementia',
# )

# clear_memory()

# batch_denoise(
#     files_Lu['Control'],
#     out_Lu / 'Control',
#     denoise_fn,
#     'Control',
# )